# H. Análisis Exploratorio de Datos (EDA) orientado a decisiones

**Proyecto:** predicción de ingreso anual superior a USD 50K — Adult Census Income (1994)

Este notebook reorganiza y profundiza únicamente los análisis técnicamente relevantes identificados en `Proyecto_Final_Mod6.ipynb`. No busca acumular gráficos: cada bloque termina con una decisión explícita de **limpieza, ingeniería de variables, modelado o negocio**.

> Pregunta rectora: **¿Qué decisión cambia como consecuencia de este resultado?**

## Decisiones que se validarán

1. Normalizar el objetivo y los faltantes antes del análisis.
2. Eliminar duplicados antes de separar entrenamiento y prueba.
3. Evitar imputar con estadísticas calculadas sobre todo el dataset; la imputación pertenece al pipeline.
4. Conservar `education-num` y descartar `education` por redundancia exacta.
5. Tratar el desbalance mediante partición estratificada y métricas adecuadas.
6. Transformar variables monetarias sesgadas y añadir indicadores de presencia de capital.
7. Manejar categorías raras y categorías no vistas sin crear columnas frágiles.
8. Evaluar de forma conjunta las variables familiares relacionadas, sin eliminarlas solo por asociación.
9. No eliminar valores extremos válidos únicamente por una regla estadística.


## 1. Preparación reproducible

Se obtiene el mismo dataset usado en el proyecto original. La limpieza aplicada aquí se limita a normalizar representaciones, sin aprender parámetros de imputación a partir del conjunto completo.


In [ ]:
%pip install -q ucimlrepo

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42


In [ ]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
df_raw = pd.concat([adult.data.features, adult.data.targets], axis=1)

print(f'Dimensiones originales: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
display(df_raw.head(3))


In [ ]:
# Normalización mínima y explícita
df = df_raw.copy()

text_cols = df.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()
    df[col] = df[col].replace({'?': pd.NA, '': pd.NA})

df['income'] = df['income'].str.rstrip('.').map({'<=50K': 0, '>50K': 1}).astype('Int64')

assert set(df['income'].dropna().unique()) == {0, 1}
assert df['income'].notna().all()
print('Objetivo normalizado: 0 = <=50K; 1 = >50K')


## 2. Calidad de datos que sí cambia el pipeline

### 2.1 Duplicados y faltantes

Los duplicados pueden terminar a ambos lados de la partición y producir una evaluación optimista. Los faltantes requieren una decisión distinta: su imputación debe aprenderse solo con entrenamiento.


In [ ]:
duplicados = int(df.duplicated().sum())
nulos = df.isna().sum().loc[lambda s: s.gt(0)].sort_values(ascending=False)

resumen_calidad = pd.DataFrame({
    'faltantes': nulos,
    'porcentaje': (100 * nulos / len(df)).round(2)
})

print(f'Duplicados exactos antes de imputar: {duplicados:,}')
display(resumen_calidad)


**Decisión de limpieza/modelado**

- Eliminar duplicados **antes** de `train_test_split` para reducir fuga entre conjuntos.
- No imputar todavía. La moda de cada columna se aprenderá dentro de un pipeline usando únicamente el conjunto de entrenamiento.
- Mantener un indicador de ausencia para `workclass`, `occupation` y `native-country`, porque “dato no informado” puede contener señal y la moda por sí sola la borraría.


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

missing_cols = ['workclass', 'occupation', 'native-country']
for col in missing_cols:
    df[f'{col}__missing'] = df[col].isna().astype('int8')

print(f'Filas después de eliminar duplicados: {len(df):,}')
print(f'Duplicados restantes: {df.duplicated().sum()}')


### 2.2 ¿La ausencia de datos cambia la tasa del objetivo?

La comparación usa diferencia en puntos porcentuales. No pretende probar causalidad; determina si conviene conservar el patrón de ausencia como variable.


In [ ]:
filas = []
base = df['income'].mean()
for col in missing_cols:
    indicador = df[col].isna()
    for estado, mascara in [('presente', ~indicador), ('faltante', indicador)]:
        filas.append({
            'variable': col,
            'estado': estado,
            'n': int(mascara.sum()),
            'tasa_>50K': df.loc[mascara, 'income'].mean(),
            'diferencia_pp_vs_total': 100 * (df.loc[mascara, 'income'].mean() - base)
        })

missing_target = pd.DataFrame(filas)
display(missing_target.style.format({'tasa_>50K': '{:.2%}', 'diferencia_pp_vs_total': '{:+.2f}'}))


**Decisión de ingeniería de variables**

Conservar los tres indicadores `__missing` y realizar imputación por moda dentro del pipeline. Si en una ejecución la tasa del objetivo entre “faltante” y “presente” resulta prácticamente igual, la utilidad del indicador se confirmará posteriormente mediante validación cruzada, no por intuición.


## 3. Variable objetivo: desbalance y criterio de éxito

La clase de interés (`income = 1`) representa aproximadamente una cuarta parte de los casos. Por eso la exactitud aislada puede premiar un modelo que casi siempre prediga la clase mayoritaria.


In [ ]:
target_summary = pd.DataFrame({
    'n': df['income'].value_counts().sort_index(),
    'proporción': df['income'].value_counts(normalize=True).sort_index()
})
display(target_summary.style.format({'proporción': '{:.2%}'}))

majority_accuracy = df['income'].value_counts(normalize=True).max()
ratio = df['income'].value_counts().max() / df['income'].value_counts().min()
print(f'Exactitud trivial de predecir siempre la mayoría: {majority_accuracy:.2%}')
print(f'Relación mayoría:minoría: {ratio:.2f}:1')


In [ ]:
X = df.drop(columns='income')
y = df['income'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

display(pd.DataFrame({
    'conjunto': ['total', 'train', 'test'],
    'filas': [len(y), len(y_train), len(y_test)],
    'tasa_>50K': [y.mean(), y_train.mean(), y_test.mean()]
}).style.format({'tasa_>50K': '{:.2%}'}))


**Decisión de modelado/negocio**

- Usar partición estratificada.
- Reportar como mínimo `recall`, `precision`, `F1`, `ROC-AUC`, `PR-AUC` y matriz de confusión; no aprobar un modelo solo por `accuracy`.
- Probar `class_weight='balanced'` dentro de validación cruzada.
- Elegir el umbral según el costo de falsos negativos y falsos positivos. Si el fin es identificar prospectos >50K, el `recall` de la clase 1 tiene prioridad, sujeto a una precisión mínima acordada con negocio.


## 4. Redundancia: `education` y `education-num`

El notebook original identificó que ambas columnas contienen la misma información en distinta representación. Aquí se verifica el mapeo uno-a-uno.


In [ ]:
edu_map = (
    df.groupby('education', dropna=False)['education-num']
      .agg(['nunique', 'min', 'max', 'size'])
      .sort_values('min')
)
display(edu_map)

assert edu_map['nunique'].max() == 1
print('Cada categoría education corresponde a un único education-num.')


**Decisión de ingeniería de variables**

Descartar `education` y conservar `education-num`: evita duplicar señal, reduce dimensionalidad tras one-hot encoding y mantiene el orden educativo. Esta decisión debe ejecutarse dentro de la preparación de variables, no después de entrenar.


## 5. Variables numéricas: forma, ceros y extremos válidos

Un resumen compacto responde tres preguntas: ¿hay asimetría?, ¿dominan los ceros?, ¿los valores están fuera del dominio documentado?


In [ ]:
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

numeric_summary = pd.DataFrame({
    'mínimo': df[numeric_cols].min(),
    'mediana': df[numeric_cols].median(),
    'media': df[numeric_cols].mean(),
    'p99': df[numeric_cols].quantile(.99),
    'máximo': df[numeric_cols].max(),
    '% ceros': 100 * df[numeric_cols].eq(0).mean(),
    'asimetría': df[numeric_cols].skew()
})
display(numeric_summary.round(2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['fnlwgt', 'capital-gain', 'capital-loss']):
    sns.histplot(np.log1p(df[col]), bins=35, ax=ax, color='#2878B5')
    ax.set_title(f'log1p({col})')
    ax.set_xlabel('Escala logarítmica')

fig.suptitle('Solo variables cuya forma cambia una decisión de transformación', y=1.04)
plt.tight_layout()
plt.show()


In [ ]:
rangos = {
    'age': (17, 90),
    'education-num': (1, 16),
    'capital-gain': (0, 99999),
    'capital-loss': (0, 4356),
    'hours-per-week': (1, 99)
}

checks = []
for col, (low, high) in rangos.items():
    fuera = (~df[col].between(low, high)).sum()
    checks.append({'variable': col, 'rango esperado': f'[{low}, {high}]', 'fuera de rango': int(fuera)})
display(pd.DataFrame(checks))


**Decisión de limpieza e ingeniería**

- No eliminar registros válidos de `hours-per-week`, `age` o montos de capital solo porque el IQR los marque como atípicos. El rango de negocio prevalece sobre una regla mecánica.
- Aplicar `log1p` a `capital-gain`, `capital-loss` y `fnlwgt` cuando el modelo sea sensible a escala/asimetría.
- Crear `has_capital_gain` y `has_capital_loss`: distinguen la presencia del evento del tamaño del monto en variables con más de 90% de ceros.
- Comparar mediante validación cruzada la transformación para modelos de árboles, que normalmente no la necesitan; conservarla como parte del pipeline para modelos lineales.


In [ ]:
for col in ['capital-gain', 'capital-loss']:
    df[f'has_{col.replace("-", "_")}'] = (df[col] > 0).astype('int8')
    df[f'log1p_{col.replace("-", "_")}'] = np.log1p(df[col])

df['log1p_fnlwgt'] = np.log1p(df['fnlwgt'])
display(df[['capital-gain', 'has_capital_gain', 'log1p_capital_gain',
            'capital-loss', 'has_capital_loss', 'log1p_capital_loss']].head())


## 6. Señal frente al objetivo: magnitud y soporte

Una tasa alta con muy pocos casos puede ser inestable. Por eso se muestran simultáneamente tasa de >50K, cantidad de registros y diferencia respecto de la tasa base.


In [ ]:
def target_rate_table(data, feature, target='income', min_support=100):
    out = (data.groupby(feature, dropna=False)[target]
               .agg(n='size', tasa_50k='mean')
               .reset_index())
    out['lift_vs_total'] = out['tasa_50k'] / data[target].mean()
    out['soporte_suficiente'] = out['n'] >= min_support
    return out.sort_values(['tasa_50k', 'n'], ascending=False)

for feature in ['occupation', 'marital-status', 'workclass']:
    print()
    print(feature)
    display(target_rate_table(df, feature).head(15).style.format({
        'tasa_50k': '{:.2%}', 'lift_vs_total': '{:.2f}×'
    }))


In [ ]:
capital_effect = pd.DataFrame({
    'segmento': ['Sin ganancia', 'Con ganancia', 'Sin pérdida', 'Con pérdida'],
    'n': [
        (df['capital-gain'] == 0).sum(), (df['capital-gain'] > 0).sum(),
        (df['capital-loss'] == 0).sum(), (df['capital-loss'] > 0).sum()
    ],
    'tasa_>50K': [
        df.loc[df['capital-gain'] == 0, 'income'].mean(),
        df.loc[df['capital-gain'] > 0, 'income'].mean(),
        df.loc[df['capital-loss'] == 0, 'income'].mean(),
        df.loc[df['capital-loss'] > 0, 'income'].mean()
    ]
})
display(capital_effect.style.format({'tasa_>50K': '{:.2%}'}))


**Decisión de modelado y negocio**

- Conservar `occupation`, `marital-status`, `workclass`, `capital-gain` y `capital-loss`: muestran separación útil del objetivo.
- No convertir tasas descriptivas en reglas deterministas ni interpretarlas causalmente.
- Para negocio, los segmentos con mayor tasa sirven para priorización, pero la decisión final debe provenir de probabilidades calibradas y un umbral asociado a costos.


## 7. Cardinalidad y categorías raras

`native-country` tiene muchas categorías y una dominante. Crear una columna manual por nivel puede producir variables ausentes cuando aparezcan nuevos datos.


In [ ]:
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']

rows = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    rows.append({
        'variable': col,
        'categorías': df[col].nunique(dropna=True),
        'categorías con <100 casos': int((counts < 100).sum()),
        '% categoría dominante': 100 * counts.iloc[0] / len(df)
    })
display(pd.DataFrame(rows).sort_values('categorías', ascending=False).round(2))


In [ ]:
country = target_rate_table(df, 'native-country', min_support=100)
display(country.head(15).style.format({'tasa_50k': '{:.2%}', 'lift_vs_total': '{:.2f}×'}))


**Decisión de ingeniería/modelado**

- Usar `OneHotEncoder(handle_unknown='ignore', min_frequency=...)` aprendido solo en entrenamiento, o agrupar niveles poco frecuentes como `Other` dentro del pipeline.
- No comparar ni actuar sobre tasas de países con soporte pequeño sin intervalos de incertidumbre.
- Validar si `native-country` mejora el desempeño fuera de muestra; su alta concentración y sensibilidad justifican una prueba de ablación.


## 8. Variables familiares relacionadas: asociación no equivale a duplicación

`relationship`, `marital-status` y `sex` están asociadas, pero no son copias exactas. Se cuantifica con V de Cramér para decidir cómo evaluarlas.


In [ ]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    denominator = max(1, min(r - 1, k - 1))
    return np.sqrt((chi2 / n) / denominator)

family_cols = ['relationship', 'marital-status', 'sex']
assoc = pd.DataFrame(index=family_cols, columns=family_cols, dtype=float)
for a in family_cols:
    for b in family_cols:
        assoc.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])

sns.heatmap(assoc, annot=True, vmin=0, vmax=1, cmap='Blues')
plt.title('V de Cramér: variables familiares y sexo')
plt.show()


**Decisión de modelado y gobernanza**

- No eliminar automáticamente ninguna de estas variables: asociación alta no implica redundancia exacta.
- Comparar por validación cruzada un modelo completo contra modelos sin `relationship`, sin `marital-status` y sin `sex`.
- Reportar desempeño por sexo y otros grupos relevantes antes de uso operativo. Si una variable sensible no agrega valor suficiente o produce brechas injustificadas, excluirla o restringir su uso.


## 9. Relación no lineal de edad, educación y horas con el objetivo

Se agrupan valores únicamente para visualizar tasas estables. Los bins no sustituyen necesariamente las variables continuas en el modelo.


In [ ]:
eda_bins = pd.DataFrame({
    'age_group': pd.cut(df['age'], bins=[16, 24, 34, 44, 54, 64, 90]),
    'hours_group': pd.cut(df['hours-per-week'], bins=[0, 20, 39, 40, 49, 60, 99]),
    'education-num': df['education-num'],
    'income': df['income']
})

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col, title in zip(
    axes,
    ['age_group', 'education-num', 'hours_group'],
    ['Edad', 'Nivel educativo', 'Horas por semana']
):
    rates = eda_bins.groupby(col, observed=True)['income'].agg(['mean', 'size']).reset_index()
    sns.lineplot(data=rates, x=col, y='mean', marker='o', ax=ax)
    ax.set_title(f'Tasa >50K por {title.lower()}')
    ax.set_ylabel('Tasa >50K')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


**Decisión de modelado**

- Conservar `age`, `education-num` y `hours-per-week`.
- No imponer linealidad sin comprobarla. Para regresión logística, comparar términos no lineales (splines o bins definidos dentro del pipeline); para árboles, conservar valores originales.
- Usar los grupos únicamente para interpretación descriptiva, evitando fuga al definir transformaciones con el conjunto de prueba.


## 10. Matriz final de decisiones

| Evidencia EDA | Decisión | Tipo | Verificación posterior |
|---|---|---|---|
| Duplicados exactos | Eliminarlos antes de dividir | Limpieza / leakage | Cero duplicados antes del split |
| Faltantes en 3 variables categóricas | Imputar moda dentro del pipeline + indicador de ausencia | Limpieza / features | Ablación de indicadores |
| Objetivo ≈ 76/24 | `stratify`, métricas por clase y prueba de pesos | Modelado | CV estratificada, PR-AUC/F1/recall |
| `education` ↔ `education-num` uno-a-uno | Eliminar `education` | Features | Confirmar mapeo en cada nueva carga |
| Capital con >90% ceros y cola larga | Indicador binario + `log1p` para modelos sensibles | Features | Comparación por CV |
| Extremos dentro del dominio | No borrarlos por IQR | Limpieza | Gates de rangos de negocio |
| Categorías raras / no vistas | Encoder robusto y agrupación aprendida en train | Modelado | Prueba con categorías nuevas |
| Variables familiares asociadas | Regularización y ablación, no borrado automático | Modelado / gobernanza | Métricas globales y por subgrupo |
| Patrones no lineales | Árboles o términos no lineales | Modelado | Comparar CV contra baseline lineal |

### Criterio de cierre del EDA

El EDA termina cuando cada hallazgo está conectado a una acción verificable. La selección definitiva de transformaciones y variables se realizará mediante validación cruzada dentro de un pipeline, manteniendo el conjunto de prueba intacto hasta la evaluación final.
